# Training 3 different RNNs

In this notebook I will train 3 different RNNs, one simple RNN, one LSTM and one GRU.

In [1]:
import numpy as np
import pandas as pd
import os
import random
from RNN_models import *
import torch
from torch import optim
from torchinfo import summary


device = torch.device("cpu")

data_folder = "../../MainProject/data/mediapipe_ugly_recordings"

In [2]:
def load_array(data_path, n_frames=30, n_features=66) -> tuple[torch.tensor, torch.tensor]:
    """
    Loads a csv-file and reshapes it into (-1, n_frames, n_features)

    Args:
        data_path: Path to the data to load
        n_frames: How many rows of data to use in shaping
        n_features: How many columns of data to use in shaping
    
    Returns:
        X, Y: The data in shape (-1, n_frames, n_features), The target as a 1darray
    """
    data = pd.read_csv(data_path)
    y = data["target"].values - 1
    x = data.drop(columns=["target"]).values
    return torch.tensor(x.reshape(-1, n_frames, n_features), dtype=torch.float32), torch.tensor(y)


def split_csvfiles(datafolder, random_seed, training_prop, validation_prop):
    csv_files = []
    for f in os.listdir(datafolder):
        if f.endswith(".csv"):
            csv_files.append(f)

    random.seed(random_seed)
    random.shuffle(csv_files)

    train_n = int(len(csv_files) * training_prop)
    val_n = int(len(csv_files) * validation_prop)

    # Split
    if validation_prop == 0:
        train_files = csv_files[:train_n]
        test_files = csv_files[train_n:]

        return train_files, test_files

    else:
        train_files = csv_files[:train_n]
        val_files = csv_files[train_n: train_n + val_n]
        test_files = csv_files[train_n + val_n:]

        return train_files, val_files, test_files

In [3]:
train_files, test_files = split_csvfiles(data_folder, random_seed=1, training_prop=0.9, validation_prop=0)

pairs = []
for file in train_files:
    path = os.path.join(data_folder, file)
    x, y = load_array(path)
    x = x.to(device)
    y = y.to(device)
    pairs.append((x, y))

print(f"Total numer of files to train on: {len(train_files)}")
print(f"Total number of files to test on: {len(test_files)}")

Total numer of files to train on: 182
Total number of files to test on: 21


### Initialize the models

In [4]:
def train_model(params: dict, training_data: tuple) -> tuple[nn.Module, dict]:
    model = params["model_type"](params["input_size"], params["hidden_size"], params["depth"], params["num_of_classification_labels"]).to(device)
    optimizer = params["optimizer"](model.parameters(), lr=params["lr"])
    loss_func = params["loss"]().to(device)
    prev_loss = np.inf
    streak = 0
    loss_increase = False
    loss_history = []
    early_stopping = False

    for epoch in range(params["epochs"]):
        total_loss = 0

        for X, Y in training_data:

            output = model(X)

            loss = loss_func(output, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
        
        loss_history.append(total_loss)

        # Check if loss increased this epoch
        if total_loss > prev_loss:
            loss_increase = True
        # Count number of times loss increase in a row
        if loss_increase:
            streak += 1
        else:
            streak = 0
        # Early stopping
        if streak == params["patience"]:
            early_stopping = True
            break

    stats = {
        "loss_history": loss_history,
        "early_stopping": early_stopping
        }
    return model, stats

### Train the three models

In [7]:
models = []
models_stats = []

params_RNN = {
    "model_type": SimpleRNNModel,
    "input_size": 66,
    "hidden_size": 64,
    "depth": 5,
    "num_of_classification_labels": 3,
    "lr": 0.0001,
    "optimizer": optim.Adam,
    "loss": nn.CrossEntropyLoss,
    "patience": 3,
    "epochs": 20
}
RNN, RNN_stats = train_model(params=params_RNN, training_data=pairs)
models.append(RNN)
models_stats.append(RNN_stats)

print(f"The final loss achieved: {RNN_stats["loss_history"][-1]}")
model_summary = summary(RNN)
print(model_summary)

The final loss achieved: 0.03829159363522194


TypeError: 'ModelStatistics' object is not callable

In [ ]:
params_LSTM = {
    "model_type": LSTMModel,
    "input_size": 66,
    "hidden_size": 64,
    "depth": 5,
    "num_of_classification_labels": 3,
    "lr": 0.0001,
    "optimizer": optim.Adam,
    "loss": nn.CrossEntropyLoss,
    "patience": 3,
    "epochs": 20
}
LSTM, LSTM_stats = train_model(params=params_LSTM, training_data=pairs)
models.append(LSTM)
models_stats.append(LSTM_stats)

print(f"The final loss achieved: {LSTM_stats["loss_history"][-1]}")
model_summary = summary(LSTM)
print(model_summary)

The final loss achieved: 0.029675445592147298


TypeError: 'ModelStatistics' object is not callable

In [ ]:
params_GRU = {
    "model_type": GRUModel,
    "input_size": 66,
    "hidden_size": 64,
    "depth": 5,
    "num_of_classification_labels": 3,
    "lr": 0.0001,
    "optimizer": optim.Adam,
    "loss": nn.CrossEntropyLoss,
    "patience": 3,
    "epochs": 20
}
GRU, GRU_stats = train_model(params=params_GRU, training_data=pairs)
models.append(GRU)
models_stats.append(GRU_stats)

print(f"The final loss achieved: {GRU_stats["loss_history"][-1]}")
model_summary = summary(GRU)
print(model_summary)

### Visualize the training

In [ ]:
from matplotlib import pyplot as plt

fig, axs = plt.subplots(1, 3, figsize=[14, 4])

for i, stats in enumerate(models_stats):
    ax = axs[i]
    ax.set_title(models[i].to_string())
    ax.plot([i for i in range(1, len(stats["loss_history"]) + 1)], stats["loss_history"])
    ax.set_xlabel("epoch")
    ax.set_ylabel("loss")

plt.show()

### Predict and evaluate

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

input = torch.zeros((len(test_files), 30, 66))
true_labels = torch.zeros(len(test_files))
for index, file in enumerate(test_files):
    path = os.path.join(data_folder, file)
    x, y = load_array(path)
    input[index, :, :] = x
    true_labels[index] = y

for i, model in enumerate(models):
    model.eval()
    with torch.no_grad():
        logits = model(input).softmax(dim=1)
        logits_index = logits.argmax(dim=1)
        print(model.to_string())
        print(logits_index + 1)
        print(true_labels.int() + 1)

        cm = confusion_matrix(true_labels, logits_index)
        ConfusionMatrixDisplay(cm).plot()
        plt.show()